In [1]:
!pip install -q transformers torch pillow requests

In [2]:
## Zero-Shot Image Classification & Retrieval with CLIP
## Using OpenAI's clip-vit-base-patch32 to measure cosine similarity between an image and arbitrary text labels without any fine-tuning.

import torch
from PIL import Image
import requests
from transformers import CLIPProcessor, CLIPModel

# 1. Load CLIP Model & Processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# 2. Download sample image
url = "https://images.unsplash.com/photo-1543466835-00a7907e9de1"
image = Image.open(requests.get(url, stream=True).raw)

# 3. Define candidate text descriptions
text_labels = ["a photo of a dog", "a photo of a cat", "a photo of a car", "a photo of a landscape"]

# 4. Preprocess and run model
inputs = processor(text=text_labels, images=image, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    # Get image-text similarity logits
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1) # Convert to probabilities

    print("--- CLIP ZERO-SHOT CLASSIFICATION ---")
    for label, prob in zip(text_labels, probs[0]):
        print(f"Label: '{label}' -> Probability: {prob.item():.4f}")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

--- CLIP ZERO-SHOT CLASSIFICATION ---
Label: 'a photo of a dog' -> Probability: 0.9992
Label: 'a photo of a cat' -> Probability: 0.0007
Label: 'a photo of a car' -> Probability: 0.0001
Label: 'a photo of a landscape' -> Probability: 0.0000


In [3]:
## Automated Image Captioning with BLIP
## Generating natural language descriptions directly from raw visual inputs using Salesforce's blip-image-captioning-base.

from transformers import BlipProcessor, BlipForConditionalGeneration

# 1. Load BLIP model and processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# 2. Prepare inputs for conditional / unconditional captioning
inputs = processor(image, return_tensors="pt")

# 3. Generate caption
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=30)
    caption = processor.decode(output[0], skip_special_tokens=True)

print("--- BLIP IMAGE CAPTIONING ---")
print("Generated Caption:", caption)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

--- BLIP IMAGE CAPTIONING ---
Generated Caption: a dog with its tongue out


In [5]:
## Visual Question Answering (VQA)
## Prompting a multimodal model with both an image and a specific question about the image content.
# Pass an explicit question as conditional text context to BLIP
from transformers import BlipProcessor, BlipForQuestionAnswering

# 1. Load BLIP model and processor specifically for VQA
# Note: This will download a different model optimized for VQA
processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")

prompt_question = "What animal is in the photo and what is it doing?"

inputs = processor(image, text=prompt_question, return_tensors="pt")

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=30)
    answer = processor.decode(output[0], skip_special_tokens=True)

print("--- VISUAL QUESTION ANSWERING (VQA) ---")
print(f"Question: '{prompt_question}'")
print(f"Answer: '{answer}'")

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.54GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

--- VISUAL QUESTION ANSWERING (VQA) ---
Question: 'What animal is in the photo and what is it doing?'
Answer: 'panting'
